In [1]:
import joblib

modelo = joblib.load("../data/modelo_emergencias_xgb_final.pkl")
print("✅ Modelo cargado correctamente.")


✅ Modelo cargado correctamente.


In [2]:
import pandas as pd
import numpy as np

# --- CONFIGURACIÓN BASE ---
FERIADOS_IMPORTANTES = [
    "2021-12-25", "2022-01-01", "2022-12-25", "2023-01-01",
    "2023-12-25", "2024-01-01", "2024-12-25", "2025-01-01"
]
poblacion_dict = pd.read_csv("../data/poblacion_provincias_ecuador_2022.csv")
def predecir_rango_emergencias(provincia, fecha_inicio, fecha_fin, modelo, poblacion_dict):
    """
    Predice el número total de emergencias esperadas para una provincia y rango de fechas.

    Parámetros:
    ------------
    provincia : str
        Nombre de la provincia (ej. 'Pichincha', 'Guayas')
    fecha_inicio : str o datetime
        Fecha inicial del rango (formato 'YYYY-MM-DD')
    fecha_fin : str o datetime
        Fecha final del rango (formato 'YYYY-MM-DD')
    modelo : sklearn Pipeline
        Modelo XGBoost entrenado y cargado desde joblib
    poblacion_dict : dict
        Diccionario con poblaciones {provincia: poblacion_2022}

    Retorna:
    ---------
    total_predicho : float
        Número total acumulado de emergencias esperadas en el rango indicado.
    df_pred : DataFrame
        Tabla con las predicciones diarias (fecha, provincia, predicción).
    """

    # Crear rango de fechas
    fechas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

    # Preparar datos
    registros = []
    poblacion = poblacion_dict.get(provincia, np.nan)

    for f in fechas:
        anio = f.year
        mes = f.month
        dia_semana = f.dayofweek  # 0 = lunes
        es_fin_semana = 1 if dia_semana in [4, 5, 6] else 0
        es_feriado = 1 if f.strftime("%Y-%m-%d") in FERIADOS_IMPORTANTES else 0
        sin_mes = np.sin(2 * np.pi * mes / 12)
        cos_mes = np.cos(2 * np.pi * mes / 12)

        registros.append({
            'provincia': provincia,
            'anio': anio,
            'mes': mes,
            'dia_semana': dia_semana,
            'es_fin_semana': es_fin_semana,
            'es_feriado': es_feriado,
            'poblacion_2022': poblacion,
            'sin_mes': sin_mes,
            'cos_mes': cos_mes
        })

    df_input = pd.DataFrame(registros)

    # Predicciones diarias
    df_input['prediccion'] = modelo.predict(df_input)

    # Resultado acumulado
    total_predicho = df_input['prediccion'].sum()

    return round(total_predicho, 2), df_input[['provincia', 'prediccion']].assign(fecha=fechas)


In [3]:
total, df_pred = predecir_rango_emergencias("Guayas", "2025-12-25", "2025-12-25", modelo, poblacion_dict)

In [4]:
print(total, df_pred)

2324.0   provincia   prediccion      fecha
0    Guayas  2324.003418 2025-12-25


In [5]:
total, df_pred = predecir_rango_emergencias("Guayas", "2025-12-20", "2025-12-31", modelo, poblacion_dict)
print(f"Predicción Guayas (20–31 dic 2025): {total:.0f} emergencias esperadas")
print(df_pred.head())

Predicción Guayas (20–31 dic 2025): 29484 emergencias esperadas
  provincia   prediccion      fecha
0    Guayas  2700.676514 2025-12-20
1    Guayas  2868.646484 2025-12-21
2    Guayas  2279.787842 2025-12-22
3    Guayas  2185.401123 2025-12-23
4    Guayas  2250.765137 2025-12-24


model_rf = joblib.load("modelo_servicios_rf.pkl")
le = joblib.load("label_encoder_servicios.pkl")


In [10]:
model_clf = joblib.load("../data/modelo_servicios_rf.pkl")
encoder  = joblib.load("../data/label_encoder_servicios.pkl")


In [11]:
import pandas as pd
import numpy as np

def predecir_total_y_cuarteto(provincia, fecha_inicio, fecha_fin,
                              modelo_reg, modelo_clf, encoder, poblacion_dict):
    """
    Predice el número total de emergencias esperadas y los 4 servicios más probables
    para una provincia y rango de fechas dados.

    Parámetros
    ----------
    provincia : str
        Nombre de la provincia.
    fecha_inicio, fecha_fin : str o datetime
        Rango de fechas a evaluar (inclusive).
    modelo_reg : modelo de regresión entrenado (XGBRegressor).
    modelo_clf : modelo de clasificación entrenado (RandomForestClassifier).
    encoder : LabelEncoder de servicios.
    poblacion_dict : dict con población por provincia.

    Devuelve
    --------
    total_predicho : float
        Total de emergencias esperadas.
    df_cuarteto : pd.DataFrame
        Tabla con los 4 servicios más probables y su probabilidad (%)
        y número estimado de emergencias.
    """

    # --- Crear rango de fechas ---
    fechas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

    # --- Construir features para el modelo de regresión ---
    df_pred = pd.DataFrame({
        'provincia': provincia,
        'anio': fechas.year,
        'mes': fechas.month,
        'dia_semana': fechas.dayofweek,
        'es_fin_semana': [1 if d in [4,5,6] else 0 for d in fechas],
        'es_feriado': [1 if d.strftime("%Y-%m-%d") in [
            "2021-12-25","2022-01-01","2022-12-25","2023-01-01",
            "2023-12-25","2024-01-01","2024-12-25","2025-01-01"
        ] else 0 for d in fechas],
        'poblacion_2022': poblacion_dict.get(provincia, np.nan),
        'sin_mes': np.sin(2 * np.pi * fechas.month / 12),
        'cos_mes': np.cos(2 * np.pi * fechas.month / 12)
    })

    # --- Predicción total diaria y agregación ---
    y_pred = modelo_reg.predict(df_pred)
    total_predicho = np.sum(y_pred)

    # --- Preparar features promedio para clasificación ---
    df_class = pd.DataFrame([{
        'provincia': provincia,
        'anio': fechas[-1].year,      # última fecha del rango
        'mes': fechas[-1].month,
        'dia_semana': fechas[-1].dayofweek,
        'es_fin_semana': 1 if fechas[-1].dayofweek in [4,5,6] else 0,
        'es_feriado': 1 if fechas[-1].strftime("%Y-%m-%d") in [
            "2021-12-25","2022-01-01","2022-12-25","2023-01-01",
            "2023-12-25","2024-01-01","2024-12-25","2025-01-01"
        ] else 0,
        'sin_mes': np.sin(2 * np.pi * fechas[-1].month / 12),
        'cos_mes': np.cos(2 * np.pi * fechas[-1].month / 12)
    }])

    # --- Cuarteto de servicios ---
    probs = modelo_clf.predict_proba(df_class)[0]
    servicios = encoder.inverse_transform(np.arange(len(probs)))
    df_cuarteto = (
        pd.DataFrame({'servicio': servicios, 'probabilidad': probs})
        .sort_values('probabilidad', ascending=False)
        .head(4)
        .reset_index(drop=True)
    )

    df_cuarteto['probabilidad'] = (df_cuarteto['probabilidad'] * 100).round(2)
    df_cuarteto['estimado'] = ((df_cuarteto['probabilidad'] / 100) * total_predicho).round(0)

    return round(total_predicho, 0), df_cuarteto


In [13]:
total, cuarteto = predecir_total_y_cuarteto(
    provincia="Guayas",
    fecha_inicio="2025-12-20",
    fecha_fin="2025-12-31",
    modelo_reg=modelo,
    modelo_clf=model_clf,
    encoder=encoder,
    poblacion_dict=poblacion_dict
)

print(f"🔹 Total estimado: {total:,.0f} emergencias")
print(cuarteto)

🔹 Total estimado: 28,904 emergencias
                servicio  probabilidad  estimado
0  Gestión De Siniestros         39.47   11408.0
1   Tránsito Y Movilidad         16.80    4856.0
2    Seguridad Ciudadana         16.16    4671.0
3      Gestión Sanitaria         15.68    4532.0
